# VB-1E-6 — Dry Run Validation of Experiment Pipeline

## Objective
This notebook performs a full end-to-end dry run of the experimental planning and execution pipeline for a simple three-slug MeCN dilution test (VB-1E-6).

The purpose is not to run chemistry, but to validate that scientific intent can be reliably transformed into an executable experiment plan.

---

## What is being tested

This workflow validates the core software stack:

- **ExperimentIntent**
  Defines the experiment at the level of scientific description (blocks, slugs, compositions)

- **ExperimentCompiler**
  Translates intent into a structured tabular representation (DataFrame), resolving conceptual definitions into concrete experimental components

- **ExperimentBuilder**
  Converts structured data into a validated `plan.json`, enforcing schema rules and execution requirements

---

## End-to-end pipeline

The full transformation being tested is:

> Scientific Intent → Compiler → DataFrame → Builder → plan.json

This represents the core abstraction layer of the autonomous flow chemistry platform.

---

## Experimental design (VB-1E-6)

This test experiment consists of:

- 3 slugs total
- Each slug contains 2 components
- Both components are MeCN (same source vial)
- Volume ratios:
  - slug_1: 20 / 80
  - slug_2: 40 / 60
  - slug_3: 50 / 50

This is a minimal system designed to validate dilution-style slug construction without introducing chemical complexity.

---

## Success criteria

The pipeline is considered successful if:

- The intent compiles without ambiguity
- The DataFrame correctly represents slug structure
- The builder produces a valid `plan.json`
- Slug grouping and volumes are preserved exactly
- No manual intervention is required between stages

---

## Notes

This is a foundational validation step. Once stable, this pipeline will be extended to:

- multi-component chemical systems
- inventory-linked reagent resolution
- dilution and concentration inference
- design-of-experiments (DoE) style campaign generation

For now, simplicity and determinism are the priority.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


from Core.inventory import Inventory
from Core.experiment_builder import ExperimentBuilder
from Core.experiment_compiler import ExperimentCompiler



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
inventory = Inventory()
builder = ExperimentBuilder()
compiler = ExperimentCompiler(inventory = inventory)

In [4]:
dilution_slugs = [
    ("slug_1", [("MeCN", 20), ("MeCN", 80)]),
    ("slug_2", [("MeCN", 40), ("MeCN", 60)]),
    ("slug_3", [("MeCN", 50), ("MeCN", 50)]),
]

In [5]:
intent = compiler.normalise(
    experiment_id="VB-1E-6",
    slug_spec=dilution_slugs,
    block_id="dilution_test"
)

intent

{'experiment_id': 'VB-1E-6',
 'blocks': [{'block_id': 'dilution_test',
   'slugs': [{'slug_id': 'slug_1',
     'composition': [('MeCN', 20), ('MeCN', 80)]},
    {'slug_id': 'slug_2', 'composition': [('MeCN', 40), ('MeCN', 60)]},
    {'slug_id': 'slug_3', 'composition': [('MeCN', 50), ('MeCN', 50)]}]}]}

In [6]:
print(json.dumps(intent, indent=2))

{
  "experiment_id": "VB-1E-6",
  "blocks": [
    {
      "block_id": "dilution_test",
      "slugs": [
        {
          "slug_id": "slug_1",
          "composition": [
            [
              "MeCN",
              20
            ],
            [
              "MeCN",
              80
            ]
          ]
        },
        {
          "slug_id": "slug_2",
          "composition": [
            [
              "MeCN",
              40
            ],
            [
              "MeCN",
              60
            ]
          ]
        },
        {
          "slug_id": "slug_3",
          "composition": [
            [
              "MeCN",
              50
            ],
            [
              "MeCN",
              50
            ]
          ]
        }
      ]
    }
  ]
}


In [7]:
df = compiler.compile(intent)

compiler.summarise_slugs(df)

,slug_id,n_components,total_volume_uL,components
0,slug_1,2,100.0,"[(MeCN, 20.0), (MeCN, 80.0)]"
1,slug_2,2,100.0,"[(MeCN, 40.0), (MeCN, 60.0)]"
2,slug_3,2,100.0,"[(MeCN, 50.0), (MeCN, 50.0)]"


In [10]:
result = builder.build_and_create(
    experiment_id=intent["experiment_id"],
    rows=df.to_dict(orient="records"),
    description="VB-1E-6 dilution validation test",
    global_conditions={
        "flowrate_ul_min": 1000.0,
        "gas_prime_s": 20.0
    },
    overwrite = True
)

result

{'plan': {'experiment_id': 'VB-1E-6',
  'description': 'VB-1E-6 dilution validation test',
  'created_at': '2026-05-01T18:58:52',
  'global_conditions': {'flowrate_ul_min': 1000.0, 'gas_prime_s': 20.0},
  'slugs': [{'slug_id': 'slug_1',
    'reaction_plan': [{'module': 'rack2',
      'vial': 1,
      'volume_uL': 20.0,
      'component': 'MeCN',
      'block_id': 'dilution_test'},
     {'module': 'rack2',
      'vial': 1,
      'volume_uL': 80.0,
      'component': 'MeCN',
      'block_id': 'dilution_test'}]},
   {'slug_id': 'slug_2',
    'reaction_plan': [{'module': 'rack2',
      'vial': 1,
      'volume_uL': 40.0,
      'component': 'MeCN',
      'block_id': 'dilution_test'},
     {'module': 'rack2',
      'vial': 1,
      'volume_uL': 60.0,
      'component': 'MeCN',
      'block_id': 'dilution_test'}]},
   {'slug_id': 'slug_3',
    'reaction_plan': [{'module': 'rack2',
      'vial': 1,
      'volume_uL': 50.0,
      'component': 'MeCN',
      'block_id': 'dilution_test'},
     {'m

In [9]:
result["summary"]

[{'slug_id': 'slug_1',
  'num_components': 2,
  'total_volume_uL': 100.0,
  'components': 'rack2:1 (20.0 uL), rack2:1 (80.0 uL)'},
 {'slug_id': 'slug_2',
  'num_components': 2,
  'total_volume_uL': 100.0,
  'components': 'rack2:1 (40.0 uL), rack2:1 (60.0 uL)'},
 {'slug_id': 'slug_3',
  'num_components': 2,
  'total_volume_uL': 100.0,
  'components': 'rack2:1 (50.0 uL), rack2:1 (50.0 uL)'}]